# Cleaning and RPM rate

26 November 2026

Notes only. Code is in src/preprocess.py and src/eda.py. Settings are in data/keep_windows.csv.

## Cleaning rules

Startup and shutdown periods were checked in MegaLogViewer and excluded with manual time limits. These transitions contained pressure and voltage readings unsuitable for the running-condition dataset.

RPM = 0 rows were first counted in all 17 processed logs. The proportions ranged from 0.06% to 11.91%, including stopped-engine periods. No whole logs were excluded based on these counts. A zero RPM reading during driving can also mean lost RPM information rather than a stopped engine.

The final row filter is RPM >= 600. This is a project scope limit for idle periods and unsuitable logs.

Rows with the same Time and segment_id are combined using the mean of the selected channels. Equal timestamps can have different readings. Combining them provides one measurement row per timestamp and avoids division by a zero time interval.

Original files in data/csv remain available.

## Final time windows

Times are in seconds. Both limits are included. An end value of inf means the end of the file. CSV uses a decimal point and commas between fields.

| Log | Start | End |
| --- | ---: | ---: |
| 05092026wires.csv | 284.206 | 665.161 |
| 05092026wires2.csv | 16.751 | 684.981 |
| 05092026wires3.csv | 74.278 | 930.843 |
| 05092026wires4.csv | 33.496 | 823.662 |
| final_log.csv | 39.890 | 670.358 |
| misfire.csv | 14.299 | 101.569 |
| misfire_driving 2.csv | 12.295 | inf |
| misfire_driving latest.csv | 15.088 | 634.549 |
| night.csv | 0 | 629.369 |
| night.csv | 1202.514 | inf |
| night2.csv | 0 | 46.308 |
| random.csv | 73.348 | inf |
| spark_wire_pt2.csv | 39.540 | 79.651 |
| synclossexperiment.csv | 0 | 793.754 |
| synclossexperiment2.csv | 41.003 | 1398.344 |
| synclossexperiment2.csv | 2755.100 | 2869.158 |
| synclossexperiment3.csv | 0 | 696.630 |
| synclossexperiment4.csv | 14.923 | inf |
| troubleshoot.csv | 0 | 629.277 |

## Manual review and result

The startup MAP readings of 144 and 165 kPa and shutdown voltage readings around 5 V were located and checked. The time windows were tightened across the logs. The earlier MAP = 291 kPa startup readings were also excluded.

PW = 0 around 285 to 286 s in 05092026wires.csv was checked against throttle release and retained. Extreme values were reviewed with operating context rather than removed by rank alone.

The final summary contains 132,224 rows across 17 files.

| Channel | Minimum | Maximum |
| --- | ---: | ---: |
| RPM | 601 | 5705 |
| MAP (kPa) | 12 | 175 |
| AFR | 10.2 | 20 |
| TPS (%) | 0 | 100 |
| IAT (°C) | 7 | 55 |
| PW (ms) | 0 | 5.9 |
| Advance (°) | 15 | 37 |
| Dwell (ms) | 2.0 | 2.1 |
| Battery V | 10.7 | 14.2 |
| Boost PSI | -12.8 | 10.9 |

The timestamp check found no repeated Time and segment_id combinations after aggregation.

## Time intervals and segments

Segment numbering uses (Time.diff() > 1).cumsum() after the row filters. A gap strictly greater than one second starts a new segment. Exactly one second does not. Segment numbers restart in each file.

Within-segment median intervals ranged from 0.058 to 0.081 s and mean intervals from 0.069 to 0.085 s. The largest interval was 0.983 s in night.csv.

A review found 44 intervals longer than 0.3 s in six logs. Many followed removed low-RPM rows. The proposed 0.3 s segment threshold was not adopted. A longer interval still permits calculation of the average rate over that interval.

## Calculated RPM rate

The original rpm/s channel exists in the CSV files, but removing RPM-zero rows does not remove interference already present in this derived channel.

Example from misfire_driving 2.csv:

| Time (s) | RPM | Recorded rpm/s | Calculated rpm_rate |
| --- | ---: | ---: | ---: |
| 18.586 | 2341 | 2550 | 5224.14 |
| 18.704 | 2539 | -21400 | 1677.97 |
| 18.766 | 2555 | 25550 | 258.06 |
| 18.852 | 2519 | -360 | -418.60 |

The recorded spikes are inconsistent with the RPM readings and are consistent with interference from lost RPM information.

rpm_rate is calculated after timestamp aggregation:

$$r_i = \frac{RPM_i-RPM_{i-1}}{Time_i-Time_{i-1}}.$$

Every consecutive retained pair in the same segment uses its actual time difference. Longer intervals produce an average rate over a longer period. The first row of each segment is NaN.

The saved CSV was checked against the EDA calculation. All 17 logs had the expected number of missing rate values: 20 missing values for 20 segments. night.csv, synclossexperiment2.csv and synclossexperiment4.csv each have two segments; the other files have one.

The recorded rpm/s column remains for comparison. Model features will use rpm_rate.

## Course connection and method choices

Chapter 3 provides the basis for Pandas selection, summaries and plotting. Chapter 4.2 introduces domain-based feature extraction; its signal examples in 4.4 illustrate features from temporal data. RPM rate is a project-specific application of that idea.

Chapter 4.5.6.1 introduces normaltest. No normality, t-test or Mann–Whitney test was run in this session. Pooled log rows contain temporal dependence and mixed operating conditions, so treating them as independent observations would not answer the current question. Histograms were used; boxplots remain a possible addition.

The main planned comparison is instantaneous sensor features versus temporal and interaction features, using the same evaluation split. Feature value will be judged by predictive performance rather than by adding statistical tests without a clear question.

Sources: ../reports/sources_2026-11-26.md.